# Model Uncertainty와 Robustness


## 학습 목표

<!-- exercise-guidance:control-2/robustness-check/objective -->
한 모델에서 설계한 제어기를 여러 실제 모델과 지연 조건에 적용해 성능이 나빠지는 지점을 찾습니다.
<!-- /exercise-guidance:control-2/robustness-check/objective -->


## 명목 제어기를 다른 실제 모델에 적용하기

<!-- exercise-guidance:control-2/robustness-check/activity-1 -->
### 시작 조건

$m\in\{.7,1,1.3\}$ kg, $b=3$ N·s/m, $k=2$ N/m입니다. K와 N은 $m=1$에서 설계한 값을 고정합니다.

### 진행 방법

1. 명목 제어기를 고정하고 세 실제 모델의 Pole과 응답을 비교하세요.
2. 각 응답의 최종값과 2% Settling Time을 구하고 가장 느린 조건을 찾으세요.

### 남길 결과

- 세 응답 곡선과 Pole·Settling Time의 비교표.
<!-- /exercise-guidance:control-2/robustness-check/activity-1 -->


In [ ]:
import numpy as np
from scipy.signal import place_poles,StateSpace,lsim
import matplotlib.pyplot as plt
A0=np.array([[0.,1.],[-2.,-3.]])
B0=np.array([[0.],[1.]]);C=np.array([[1.,0.]])
K=place_poles(A0,B0,[-2.,-4.]).gain_matrix;N=8.
t=np.linspace(0,6,1201)
for m in [.7,1.,1.3]:
    A=np.array([[0.,1.],[-2/m,-3/m]]);B=np.array([[0.],[1/m]])
    poles=np.linalg.eigvals(A-B@K)
    _,y,_=lsim(StateSpace(A-B@K,B*N,C,0),np.ones(len(t)),t)
    print('mass',m,'poles',poles);plt.plot(t,y,label=f'm={m}')
plt.xlabel('Time [s]');plt.ylabel('Position');plt.legend();plt.grid()


## 결과 기록·자가 점검

<!-- exercise-guidance:control-2/robustness-check/activity-1-record -->
**질량:**

**Pole:**

**최종값:**

**Settling Time:**

### 확인할 점

- 실제 모델마다 K를 다시 설계하지 않았는가?
- Settling Time 이후에도 2% 띠 안에 남는지 확인했는가?
<!-- /exercise-guidance:control-2/robustness-check/activity-1-record -->


## 질량·마찰·강성을 함께 바꾸기

<!-- exercise-guidance:control-2/robustness-check/activity-2 -->
### 시작 조건

m∈{.7,1,1.3}, b∈{2,3,4}, k∈{1,2,3}의 27개 조합입니다. $K=\begin{bmatrix}6&3\end{bmatrix}$, $N=8$은 고정합니다.

### 진행 방법

1. 각 조합의 $A-BK$ 고유값에서 실수부의 최댓값을 구하세요. 이 값이 음수인 모델 수와 0 이상인 모델 수를 기록하세요.
2. 실수부 최댓값이 가장 큰 조합을 찾으세요. 각 조합에서 $y_{ss}=-C(A-BK)^{-1}BN$도 구하고, $|1-y_{ss}|$가 가장 큰 조합을 따로 찾으세요.
3. 선택한 두 조합의 $m,b,k$·최대 Pole 실수부·Steady State 오차를 표로 적고 Unit Step을 같은 시간축에 그리세요.
4. 이번 계산에서 검사한 매개변수 값과 그 사이의 검사하지 않은 값을 구분해 적으세요. 구간 전체의 안정성을 판단하려면 학습자료의 Characteristic Polynomial에서 어느 계수 조건을 살펴봐야 하는지 설명하세요.

### 남길 결과

- 27개 조건의 안정성 요약과 두 극단 조건의 응답.
<!-- /exercise-guidance:control-2/robustness-check/activity-2 -->


In [ ]:
from itertools import product
records=[]
for m,b,k in product([.7,1.,1.3],[2.,3.,4.],[1.,2.,3.]):
    A=np.array([[0.,1.],[-k/m,-b/m]]);B=np.array([[0.],[1/m]])
    aa=A-B@K;pole=np.max(np.linalg.eigvals(aa).real)
    final=float((-C@np.linalg.solve(aa,B*N)).item())
    records.append((m,b,k,pole,abs(1-final)))
print('closest pole:',max(records,key=lambda r:r[3]))
print('largest error:',max(records,key=lambda r:r[4]))
# TODO: 두 모델의 Step 응답을 같은 축에 추가하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:control-2/robustness-check/activity-2-record -->
**m,b,k:**

**최대 Pole 실수부:**

**Steady-State Error:**

### 확인할 점

- Pole은 A가 아니라 $A-BK$에서 계산했는가?
- 27개 조합의 Pole 계산 결과와 전체 파라미터 범위에 대한 해석적 근거를 구분했는가?
<!-- /exercise-guidance:control-2/robustness-check/activity-2-record -->


## 입력 지연과 포화가 있는 경우

<!-- exercise-guidance:control-2/robustness-check/activity-3 -->
### 시작 조건

명목 모델에서 `dt=.002` s, 6 s 동안 상태를 Euler 적분합니다. 입력 지연은 0,.08,.2 s이고 입력 한계는 ±4입니다.

### 진행 방법

1. 실습용 노트북의 함수로 세 지연의 출력·입력을 그리세요.
2. dt를 절반으로 줄여 주요 응답이 비슷한지 확인하세요.
3. 선형 Pole의 안정성 판정만으로 지연·포화가 있는 구현을 평가하기 어려운 이유를 설명하세요.

### 남길 결과

- 출력·입력 곡선, RMS 추종 오차(2 s 이후)·입력 포화 비율(전체 0–6 s 기록)과 적분 간격 비교.
<!-- /exercise-guidance:control-2/robustness-check/activity-3 -->


In [ ]:
def delayed(delay,dt=.002):
    times=np.arange(0,6+dt/2,dt);x=np.zeros(2);commands=[];ys=[];us=[]
    lag=int(round(delay/dt))
    for _ in times:
        command=float(np.clip(N-(K@x).item(),-4,4));commands.append(command)
        u=commands[-1-lag] if len(commands)>lag else 0.
        ys.append(x[0]);us.append(u)
        x=x+dt*(A0@x+B0[:,0]*u)
    return times,np.array(ys),np.array(us)
for delay in [0.,.08,.2]:
    tt,yy,uu=delayed(delay);keep=tt>=2;print(delay,np.sqrt(np.mean((1-yy[keep])**2)),np.mean(np.abs(uu)>=4-1e-8))
    plt.plot(tt,yy,label=str(delay))
plt.xlabel('Time [s]');plt.ylabel('Position');plt.legend();plt.grid()
# TODO: 입력 그래프와 dt=.001의 결과 비교를 추가하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:control-2/robustness-check/activity-3-record -->
**지연:**

**dt:**

**RMS 추종 오차:**

**입력 포화 비율:**

### 확인할 점

- 지연은 명령 기록에서 과거 입력을 가져오는 방식인가?
- 시간 간격 차이를 모델 불안정과 혼동하지 않았는가?
<!-- /exercise-guidance:control-2/robustness-check/activity-3-record -->


## 공식 예제·데이터

<!-- exercise-guidance:control-2/robustness-check/resources -->
- [SciPy · Pole placement](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.place_poles.html)
<!-- /exercise-guidance:control-2/robustness-check/resources -->
